In [75]:
import os
import gc
os.chdir(r'C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
from src.utils import read_csv_file
import config
from sklearn.model_selection import StratifiedKFold
from pytabkit import RealMLP_TD_Classifier
from sklearn.metrics import log_loss, balanced_accuracy_score
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

In [76]:
train = read_csv_file('data/raw/train.csv')

Reading file from: data/raw/train.csv


In [77]:
X = train.drop(['id', 'irrigation_need'], axis=1)
y = train.iloc[:, -1].map(config.TARGET_MAP)

In [78]:
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

In [79]:
params = {
    "device": 'cuda',
    "random_state": config.SEED,
    "n_cv": 1,
    "n_refit": 0,
    "n_repeats": 1,
    "val_fraction": 0.0,
    "n_threads": 12,
    "verbosity": 1,
    "val_metric_name": '1-balanced_accuracy',
    "n_epochs": 5
}

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEED)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
	print(f"\nFold {fold+1}")

	X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
	y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

	model = RealMLP_TD_Classifier(**params)
	model.fit(X_train, y_train, X_valid, y_valid, cat_col_names=cat_cols)
	y_pred = model.predict(X_valid)

	print(f'Bacc: {balanced_accuracy_score(y_valid, y_pred)}')
